In [5]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image

In [3]:
dataset_path = "dataset"   

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30

In [5]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.3,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

In [6]:
train_gen = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

Found 1668 images belonging to 17 classes.


In [7]:
val_gen = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

Found 408 images belonging to 17 classes.


In [10]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

29084464/29084464 [==============================] - 32s 1us/step


In [11]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

In [12]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(17, activation="softmax")
])

In [13]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [15]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early_stop, lr_scheduler]
)

Epoch 1/30
105/105 [==============================] - 169s 2s/step - loss: 3.0752 - accuracy: 0.1025 - val_loss: 2.6446 - val_accuracy: 0.2010 - lr: 1.0000e-04
Epoch 2/30
105/105 [==============================] - 152s 1s/step - loss: 2.7176 - accuracy: 0.1691 - val_loss: 2.5287 - val_accuracy: 0.2083 - lr: 1.0000e-04
Epoch 3/30
105/105 [==============================] - 140s 1s/step - loss: 2.5358 - accuracy: 0.2128 - val_loss: 2.4725 - val_accuracy: 0.2255 - lr: 1.0000e-04
Epoch 4/30
105/105 [==============================] - 148s 1s/step - loss: 2.4183 - accuracy: 0.2482 - val_loss: 2.4024 - val_accuracy: 0.2696 - lr: 1.0000e-04
Epoch 5/30
105/105 [==============================] - 155s 1s/step - loss: 2.2952 - accuracy: 0.2644 - val_loss: 2.3430 - val_accuracy: 0.2500 - lr: 1.0000e-04
Epoch 6/30
105/105 [==============================] - 153s 1s/step - loss: 2.2271 - accuracy: 0.2998 - val_loss: 2.2962 - val_accuracy: 0.2892 - lr: 1.0000e-04
Epoch 7/30
105/105 [====================

In [22]:
model.save("fracture_model_densenet121.h5")
print("Model saved successfully!")

Model saved successfully!


In [6]:
from tensorflow.keras.models import load_model

model = load_model("fracture_model_densenet121.h5")
print("Model loaded successfully!")

Model loaded successfully!


In [7]:
class_names = list(train_gen.class_indices.keys())

import pickle
with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

NameError: name 'train_gen' is not defined

In [3]:
with open("class_names.pkl", "rb") as f:
    class_names = pickle.load(f)

NameError: name 'pickle' is not defined

In [4]:
train_acc = history.history['accuracy'][-1] * 100
val_acc = history.history['val_accuracy'][-1] * 100

print("Training Accuracy: {:.2f}%".format(train_acc))
print("Validation Accuracy: {:.2f}%".format(val_acc))

NameError: name 'history' is not defined

In [8]:
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.show()

NameError: name 'history' is not defined

In [9]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_from_path(img_path):

    IMG_SIZE = 224

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    class_index = np.argmax(prediction)
    confidence = np.max(prediction)

    return class_names[class_index], confidence

In [ ]:
def get_severity(conf):

    if conf < 0.4:
        return "Mild"
    elif conf < 0.75:
        return "Moderate"
    else:
        return "Severe"

In [11]:
img_path = input("Enter image path: ")

label, conf = predict_from_path(img_path)

print("\nPrediction:", label)
print("Confidence: {:.2f}%".format(conf * 100))
print("Severity:", get_severity(conf))

1/1 [==============================] - 2s 2s/step


NameError: name 'class_names' is not defined